# Validação Agrupada (`GroupKFold`) e Bloqueios Anti-Vazamento

Neste notebook auditamos visualmente a separação estrita de sujeitos/pacientes entre **Desenvolvimento e Holdout** e entre **Treino e Validação**, executamos um experimento com holdout isolado e testamos bloqueios ao vivo.

In [1]:
# Instalação automática apenas quando executado no Google Colab
import subprocess, sys
if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/aleerangel/experimental-ml-framework.git",
    ])

In [2]:
# Helper de exibição rica (funciona em Jupyter/Colab e em Python puro)
def exibir_markdown(texto: str) -> None:
    try:
        from IPython.display import Markdown, display
        display(Markdown(texto))
    except ImportError:
        print(texto)

def exibir_svg(svg_bytes: bytes | str) -> None:
    svg_str = svg_bytes.decode("utf-8") if isinstance(svg_bytes, bytes) else svg_bytes
    try:
        from IPython.display import SVG, display
        display(SVG(svg_str))
    except ImportError:
        print(f"[Figura SVG gerada: {len(svg_str)} caracteres]")

## 1. Auditoria Visual de Grupos no `PartitionPlan` (`CV_AND_HOLDOUT`)

In [3]:
from pathlib import Path
import tempfile
from experimental_ml.core.enums import TaskType
from experimental_ml.core.findings import ExperimentalMLError
from experimental_ml.data import (
    DatasetSchemaSpec,
    create_preparation_plan,
    load_dataset,
    profile_dataset,
    sanitize_dataset,
)
from experimental_ml.features import FeatureScenario, ScenarioMode
from experimental_ml.models import DummyBaselineModel, ModelCapabilities, ModelFamily, ModelSpec
from experimental_ml.models.preprocessing import (
    PreprocessingStepSpec,
    ScalerType,
    TwoLayerPreprocessingSpec,
    validate_two_layer_preprocessing,
)
from experimental_ml.orchestration import load_cell_artifacts, run_experiment_matrix
from experimental_ml.validation import (
    EvaluationRegime,
    PartitionConfig,
    PartitionStrategy,
    build_partition_plan,
    validate_partition_plan,
)

dados = {
    "paciente_id": [f"P{g}" for g in range(1, 7) for _ in range(4)],
    "biomarcador_a": [float(i) * 0.1 for i in range(24)],
    "biomarcador_b": [float(24 - i) * 0.2 for i in range(24)],
    "diagnostico": [0 if g <= 3 else 1 for g in range(1, 7) for _ in range(4)],
}
schema = DatasetSchemaSpec(
    target_col="diagnostico",
    task_type=TaskType.BINARY_CLASSIFICATION,
    group_col="paciente_id",
)
bundle = load_dataset(dados, schema)
profile = profile_dataset(bundle)
view = sanitize_dataset(bundle, profile, create_preparation_plan(profile, rules=[]))

part_plan = build_partition_plan(
    view,
    PartitionConfig(
        strategy=PartitionStrategy.GROUP_KFOLD,
        regime=EvaluationRegime.CV_AND_HOLDOUT,
        n_splits=3,
        holdout_fraction=0.33,
        group_col="paciente_id",
        master_seed=42,
    ),
)
validate_partition_plan(part_plan, view)

id_to_group = {oid: grp for oid, grp in zip(view.observation_ids, view.columns["paciente_id"])}
dev_groups = sorted({id_to_group[oid] for oid in part_plan.development_ids})
hold_groups = sorted({id_to_group[oid] for oid in part_plan.holdout_ids})
print(f"Grupos Desenvolvimento: {dev_groups}")
print(f"Grupos Holdout Isolado: {hold_groups}")
for f in part_plan.folds:
    tr_g = sorted({id_to_group[oid] for oid in f.train_ids})
    va_g = sorted({id_to_group[oid] for oid in f.val_ids})
    print(f"  Dobra {f.fold_index}: Treino={tr_g} | Validação={va_g}")

Grupos Desenvolvimento: ['P1', 'P2', 'P5', 'P6']
Grupos Holdout Isolado: ['P3', 'P4']
  Dobra 0: Treino=['P2', 'P5'] | Validação=['P1', 'P6']
  Dobra 1: Treino=['P1', 'P2', 'P6'] | Validação=['P5']
  Dobra 2: Treino=['P1', 'P5', 'P6'] | Validação=['P2']


## 2. Execução e Métricas com Holdout Isolado

In [4]:
from experimental_ml.statistics import consolidate_descriptive

tmp_dir = Path(tempfile.mkdtemp(prefix="expml_nb03_"))
cen = FeatureScenario(
    scenario_id="bio", name="Biomarcadores", description="A e B",
    scientific_justification="Base clínica", mode=ScenarioMode.FIXED,
    selected_features=("biomarcador_a", "biomarcador_b"),
)
mod = ModelSpec(
    model_id="dummy_freq", display_name="Dummy", family=ModelFamily.BASELINE,
    estimator_factory=lambda seed=None: DummyBaselineModel(strategy="most_frequent"),
    capabilities=ModelCapabilities(supports_predict_proba=True),
    description="Dummy", scientific_justification="Referência",
)
state = run_experiment_matrix(
    model_specs=[mod], scenarios=[cen], partition_plan=part_plan,
    sanitized_view=view, run_dir=tmp_dir / "runs", metrics=("accuracy", "f1"), master_seed=42, overwrite=True,
)
cells = [load_cell_artifacts(cid, tmp_dir / "runs")[0] for cid in state.cell_statuses]
cons = consolidate_descriptive(cells, total_folds_per_config=len(part_plan.folds))
summary = next(iter(cons.configurations.values()))
for m_name, m_stats in summary.aggregated_metrics.items():
    print(f"  Métrica '{m_name}': média={m_stats.mean:.4f}, std={m_stats.std:.4f}")

  Métrica 'accuracy': média=0.1667, std=0.2887
  Métrica 'f1': média=0.0000, std=0.0000


## 3. Laboratório de Bloqueios ('O que acontece se...')

In [5]:
# Bloqueio 1: GROUP_KFOLD sem informar group_col
try:
    PartitionConfig(strategy=PartitionStrategy.GROUP_KFOLD, regime=EvaluationRegime.CV_ONLY, n_splits=3)
    raise AssertionError("Deveria ter bloqueado GROUP_KFOLD sem group_col!")
except ExperimentalMLError as exc:
    print("[BLOQUEIO ESPERADO 1]:", exc)

# Bloqueio 2: Escalonamento idêntico duplicado nas 2 camadas (Cenário + Modelo)
try:
    validate_two_layer_preprocessing(
        TwoLayerPreprocessingSpec(
            layer1_steps=(PreprocessingStepSpec(step_id="s1", transformer_type=ScalerType.STANDARD.value),),
            layer2_steps=(PreprocessingStepSpec(step_id="s2", transformer_type=ScalerType.STANDARD.value),),
        )
    )
    raise AssertionError("Deveria ter bloqueado escalonamento duplicado!")
except ExperimentalMLError as exc:
    print("[BLOQUEIO ESPERADO 2]:", exc)

[BLOQUEIO ESPERADO 1]: GROUP_KFOLD strategy requires 'group_col' to be specified
[BLOQUEIO ESPERADO 2]: Accidental duplication detected (CA-E3-02): transformer_type 'STANDARD' is declared in both Layer 1 and Layer 2.
